# 数据增强原理与实践

本 notebook 讲解深度学习里的**数据增强（Data Augmentation）**：是什么、为什么能治过拟合、有哪些类别、你工程里用了哪些。所有可视化都用你工程 `data/` 里的真实 CIFAR-10 图片生成。

## 一句话

数据增强 = 每次读取训练图片时随机改动一下（裁剪/翻转/变色/擦除），让模型每次看到的"同一张图"都不一样，等效于无限扩容训练集。

## 目录

1. 核心思想：为什么增强能治过拟合
2. 四大类增强：几何 / 像素 / 遮挡 / 自动搜索
3. 你工程 `data.py` 的增强流水线对照
4. 为什么是这个顺序：PIL 与 Tensor 的格式要求
5. 为什么验证集不做随机增强
6. 关键原则与陷阱

## 1. 核心思想：为什么增强能治过拟合

### 过拟合的本质

模型把训练集的**噪声和细节**也记住了，而不是学到**通用特征**。表现为：训练 loss 一直降，但验证 loss 先降后升。

### 增强怎么治

- **等效扩容**：45000 张图，每张每次读出来都不一样，等效于"无限多张图"
- **打破捷径**：模型不能靠"猫耳尖尖的就是猫"这种局部捷径，因为增强可能把耳朵裁掉或擦掉
- **学不变特征**：翻转/变色后的猫还是猫，模型被迫学"猫的本质"而非"某只猫的样子"

### 直觉

教小孩认猫：只给他看 5 张猫的照片，他会死记这 5 张（过拟合）；给他看 5 张但每次翻转/裁剪/变色，等效看了几百张不同的猫，他被迫学"猫的通用特征"（泛化）。

## 2. 四大类增强

### 几何变换类：改"位置和方向"

- **RandomCrop**：随机裁剪（先填边再裁回原尺寸）。模拟"目标在画面不同位置"
- **RandomHorizontalFlip**：随机水平翻转。模拟"目标朝左/朝右"
- 旋转/缩放/平移也属此类

适用前提：变换后**标签不变**。猫翻转还是猫，但数字 6 翻转变成 9 就不能翻转。

### 像素变换类：改"外观"

- **ColorJitter**：亮度/对比度/饱和度/色调随机抖动
- 模拟"不同光照/不同相机/不同天气"
- 你工程没直接用 ColorJitter，但 RandAugment 内部包含这类操作

### 遮挡类：治"依赖局部"

- **RandomErasing** / **Cutout**：随机擦掉一小块
- 迫使模型不能依赖某个局部特征（如"猫耳"），必须看整体
- 你工程用了 RandomErasing(p=0.25)

### 自动搜索类：不用手调

- **RandAugment** / **AutoAugment**：从一组变换里自动随机选 N 种、强度由 magnitude 控制
- 不用手动挑变换组合，一个参数控制总强度
- 你工程用了 RandAugment(num_ops=2, magnitude=9)

## 3. 你工程 `data.py` 的增强流水线

你工程的训练增强（正则化版）一共 5 步，按顺序执行：

| 顺序 | 操作 | 类别 | 作用 |
|------|------|------|------|
| 1 | RandomCrop(32, padding=4) | 几何 | 模拟目标位置变化 |
| 2 | RandomHorizontalFlip(0.5) | 几何 | 模拟朝向变化 |
| 3 | RandAugment(2, 9) | 自动 | 随机选 2 种像素/几何变换 |
| 4 | ToTensor + Normalize | 预处理 | 转 Tensor + 标准化 |
| 5 | RandomErasing(0.25) | 遮挡 | 25% 概率擦一块 |

对比旧版：旧版只有第 1、2 步（基础几何增强），新版加了 RandAugment + RandomErasing，增强强度显著提升。

下面两节深入讲两个关键问题：**为什么是这个顺序** 和 **为什么验证集不增强**。

## 4. 为什么是这个顺序：PIL 与 Tensor 的格式要求

数据增强的顺序不是随便排的，由**每个变换对输入数据格式的要求**决定。

### 两种数据格式

- **PIL Image**：PIL 库的图片对象，内部是 H x W x C 的整数像素矩阵（0-255）。传统图像处理库（裁剪/翻转/颜色调整）在这上面操作最自然。
- **torch.Tensor**：形状 C x H x W 的浮点张量（0-1 或标准化后）。PyTorch 的某些变换需要这种格式。

### 每个变换的格式要求

| 变换 | 要求输入 | 原因 |
|------|----------|------|
| RandomCrop / Flip | PIL 或 Tensor | 两者都支持 |
| **RandAugment** | **只支持 PIL** | 内部用 PIL.ImageEnhance 等 PIL API |
| ToTensor | PIL | 把 PIL(0-255, HWC) 转成 Tensor(0-1, CHW) |
| Normalize | **只支持 Tensor** | 浮点数学运算 |
| **RandomErasing** | **只支持 Tensor** | 在张量上索引置值 |

### 顺序因此被锁死

1. **PIL 阶段**（ToTensor 之前）：所有 PIL-only 变换必须在这做 -> RandomCrop、Flip、RandAugment
2. **ToTensor**：格式转换的桥梁
3. **Tensor 阶段**（ToTensor 之后）：所有 Tensor-only 变换必须在这做 -> Normalize、RandomErasing

如果把 RandomErasing 放到 ToTensor 之前（PIL 阶段），它会报错——PIL 没有 tensor 的索引操作。如果把 RandAugment 放到 ToTensor 之后（Tensor 阶段），它也会报错——需要 PIL 对象。

### 验证一下（可以自己跑）

```python
# 这个会报错：RandAugment 需要 PIL，但 ToTensor 后已经是 Tensor
bad = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandAugment(num_ops=2, magnitude=9),  # 报错：期望 PIL Image
])

# 这个也会报错：RandomErasing 需要 Tensor，但还没 ToTensor
bad2 = transforms.Compose([
    transforms.RandomErasing(p=1.0),  # 报错：期望 Tensor
    transforms.ToTensor(),
])
```

### 一句话

**顺序 = 格式要求的拓扑序**：PIL-only 的必须在前，Tensor-only 的必须在后，ToTensor 是中间那座桥。这不是风格选择，是硬性约束。

## 5. 为什么验证集/测试集不做随机增强

### 评估要回答的问题

训练时增强是"故意的随机性"——让模型看到更多样的数据。但评估时要回答的是：**"这个模型在新数据上表现如何"**。

### 增强会破坏评估

如果验证集也随机增强：

- 同一张图每次评估输入不同（翻转/变色/擦除都不同）
- 模型输出不同 -> 准确率每次波动
- 你无法判断：模型真有 80% 还是这次增强碰巧容易/难
- 评估**失去可复现性**和**可比性**

### 更本质：验证集模拟真实使用场景

真实使用时用户给的是**原始图片**（不会翻转、不会擦除），所以验证也要用原始图片（只做 ToTensor + Normalize 这种确定性预处理）。增强后的图不是"真实场景的图"，用它评估测不出真实泛化能力。

### 如果验证集增强会怎样

假设模型真实水平 80%。验证集增强后：

- 某次增强恰好把难分类的猫耳朵擦掉 -> 准确率掉到 75%
- 某次增强恰好没影响关键特征 -> 准确率 81%
- 每次评估结果不同，你不知道模型到底是 75% 还是 81%

### 什么算"确定性预处理"

验证集**可以**做的：`ToTensor`（格式转换）+ `Normalize`（标准化）—— 这两步**每次结果一样**（同一张图 -> 同一个 Tensor），不破坏可复现性。

验证集**不能**做的：任何带"Random"的操作（RandomCrop / Flip / RandAugment / RandomErasing）—— 这些每次结果不同。

### 你工程的对照

```python
# 训练集：5 步，含 4 个随机增强
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),       # 随机
    transforms.RandomHorizontalFlip(p=0.5),     # 随机
    transforms.RandAugment(num_ops=2, magnitude=9),  # 随机
    transforms.ToTensor(),                       # 确定
    transforms.Normalize(mean=..., std=...),     # 确定
    transforms.RandomErasing(p=0.25),            # 随机
])

# 验证集/测试集：只有 2 步，全是确定性操作
evaluation_transform = transforms.Compose([
    transforms.ToTensor(),                       # 确定
    transforms.Normalize(mean=..., std=...),    # 确定
])
```

### 一句话

**训练集增强 = 扩容（要随机）；验证集不增强 = 保证可复现 + 模拟真实场景（要确定）。一个要随机，一个要确定，目的不同所以做法相反。**

## 6. 关键原则与陷阱

### 原则 1：验证集/测试集绝不增强

评估要可复现。如果验证集也随机增强，同一模型每次评估准确率都会波动，无法判断模型好坏。验证/测试只做确定性预处理（ToTensor + Normalize）。

### 原则 2：变换后标签不能变

- CIFAR-10 猫翻转还是猫 -> 可以水平翻转
- MNIST 数字 6 翻转变成 9 -> **不能**翻转
- 选择增强前先想：这个变换会不会改变语义？

### 原则 3：强度别过头

- 增强太弱 -> 没效果（旧版只有 Crop+Flip 就偏弱）
- 增强太强 -> 欠拟合（把猫擦掉 80% 模型根本认不出）
- 你工程 magnitude=9 是中等偏强，RandomErasing 擦除面积 2%-20% 也合理

### 原则 4：组合使用，多角度覆盖

单一增强效果有限。你工程同时用几何 + 自动 + 遮挡三类，覆盖"位置/外观/局部"三个维度，比单一增强强得多。

### 陷阱：增强只在训练时生效

`transforms` 在 DataLoader 读取时执行。评估时用的是另一个 `evaluation_transform`（不增强）。如果你不小心给测试集也加了随机增强，评估结果会失真——这是常见错误。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms

# 读取一张 CIFAR-10 原始图片。
raw_dataset = datasets.CIFAR10(root=Path('data'), train=True, download=True)
original_image, label = raw_dataset[0]

# 为了观察效果，这里把 RandomErasing 的概率设为 1；
# 正式训练仍使用 data.py 中的 p=0.25。
visual_transforms = {
    '原图': transforms.Compose([]),
    'Crop + Flip': transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(p=1.0),
    ]),
    'RandAugment': transforms.RandAugment(num_ops=2, magnitude=9),
    'RandomErasing': transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomErasing(p=1.0, scale=(0.10, 0.25)),
    ]),
}

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for axis, (name, transform) in zip(axes, visual_transforms.items()):
    transformed = transform(original_image)
    if torch.is_tensor(transformed):
        transformed = transformed.permute(1, 2, 0).clamp(0, 1).numpy()
    axis.imshow(transformed)
    axis.set_title(name)
    axis.axis('off')

fig.suptitle(f'CIFAR-10 数据增强示例：真实标签 = {raw_dataset.classes[label]}')
plt.tight_layout()
plt.show()


## 7. 实践观察：增强前后到底改变了什么

运行上面的可视化单元格后，可以从四个角度观察：

- **Crop + Flip** 改变目标的位置和朝向，但通常不改变类别；
- **RandAugment** 改变颜色、对比度或局部几何特征，模拟不同拍摄条件；
- **RandomErasing** 删除局部信息，检查模型是否能根据剩余区域识别类别；
- 图片尺寸仍然是 $32 \times 32$，通道数仍然是 3，改变的是像素内容而不是张量形状。

### 如何判断增强是否合理

增强后的图片应该满足两个条件：

1. 人仍然能够大致判断它原来的类别；
2. 它不能总是和原图完全一样。

如果增强后已经看不出目标是什么，说明增强太强，模型得到的监督信号会变差。比如对 CIFAR-10 使用水平翻转通常合理，但对数字识别任务随意旋转可能会把 6 变成 9。

## 8. Mixup：不只改变图片，也改变标签

前面介绍的 Crop、Flip、RandAugment 和 RandomErasing 都属于**标签保持型增强**：图片变了，但标签仍然是原来的类别。Mixup 不同，它会同时混合图片和标签。

取两张图片 $x_a$、$x_b$ 及标签 $y_a$、$y_b$，随机产生混合比例 $\lambda$：

$$
\tilde{x} = \lambda x_a + (1-\lambda)x_b
$$

$$
\tilde{y} = \lambda y_a + (1-\lambda)y_b
$$

例如 $\lambda=0.8$ 时，混合样本包含约 80% 的第一张图片信息和 20% 的第二张图片信息，损失也按同样比例计算：

$$
L = \lambda L(\hat{y}, y_a) + (1-\lambda)L(\hat{y}, y_b)
$$

当前 Tiny ViT 在 `training.py` 的训练 batch 中使用 `mixup_alpha=0.1`。Mixup 的目标是让模型学习更平滑的决策边界，减少记住某张训练图片背景、纹理或局部像素的倾向。

### 数据增强与 Mixup 的区别

| 方法 | 图片是否改变 | 标签是否改变 | 主要作用 |
|------|--------------|--------------|----------|
| RandomCrop / Flip | 是 | 否 | 模拟位置和方向变化 |
| RandAugment | 是 | 否 | 模拟外观和几何变化 |
| RandomErasing | 是 | 否 | 减少局部特征依赖 |
| Mixup | 是，两张图片混合 | 是，标签也混合 | 让分类边界更平滑 |

由于训练阶段的图片经过增强，尤其是 Mixup，训练准确率不一定高于验证准确率。验证集和测试集必须使用原始图片，只做确定性的 `ToTensor + Normalize`，这样才能衡量模型对真实未见样本的泛化能力。

## 9. 自检问题

1. 为什么 `RandAugment` 要放在 `ToTensor` 前面，而 `RandomErasing` 要放在后面？
2. 为什么验证集不能使用 `RandomCrop` 或 `RandomErasing`？
3. 哪些增强会保持标签不变？哪些增强会同时改变标签？
4. 如果增强后人已经无法判断原类别，应该调整哪个方向的参数？
5. 为什么使用 Mixup 后，`train acc` 可能低于 `val acc`？


## 小结

### 四大类增强

| 类别 | 代表操作 | 治什么 |
|------|----------|--------|
| 几何 | RandomCrop / Flip | 位置/朝向过拟合 |
| 像素 | ColorJitter / RandAugment 内的 | 光照/外观过拟合 |
| 遮挡 | RandomErasing / Cutout | 依赖局部特征 |
| 自动 | RandAugment / AutoAugment | 不用手调，一个参数控总强 |

### 顺序由格式决定

PIL-only 变换（RandAugment）在 ToTensor 之前，Tensor-only 变换（RandomErasing）在 ToTensor 之后。顺序是硬约束。

### 验证集不增强

训练要随机（扩容），验证要确定（可复现 + 模拟真实场景）。目的不同，做法相反。

### 一句话

**数据增强 = 让模型每次看到的"同一张图"都不一样，等效无限扩容训练集，迫使模型学通用特征而非死记样本。** 它和 Dropout / Label Smoothing / Early Stopping / Weight Decay 互补——增强治"死记样本"，其他治"学过头/过度自信/权重过大"，五个方向一起压过拟合。